In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
import json
from collections import defaultdict



In [2]:
sample_df = pd.read_csv('data/protein/AHA-1_SYS135.csv')

In [4]:
sample_df[sample_df["Cell-name"] == "ABalaaaalar"]

,TF_strain,Time,Cell-name,Raw-expression,Adjustment-expression
42490,AHA-1_SYS135,223,ABalaaaalar,2.10,2.16
43040,AHA-1_SYS135,224,ABalaaaalar,0.00,0.00
43589,AHA-1_SYS135,225,ABalaaaalar,0.00,0.00
44139,AHA-1_SYS135,226,ABalaaaalar,0.37,0.37
44685,AHA-1_SYS135,227,ABalaaaalar,2.69,3.10
45237,AHA-1_SYS135,228,ABalaaaalar,0.00,0.00
45787,AHA-1_SYS135,229,ABalaaaalar,0.62,0.62
46339,AHA-1_SYS135,230,ABalaaaalar,0.00,0.00
46886,AHA-1_SYS135,231,ABalaaaalar,0.45,0.45
47434,AHA-1_SYS135,232,ABalaaaalar,1.26,1.26


In [48]:
common_lineage_names = set(sample_df['Cell-name'].unique())
len(common_lineage_names)

1232

In [56]:
# name mapping
def map_names(did):
    """ Re-map cells to use their 'name' given their 'did'. Only applies to a
        few select cells where the tracker uses their 'name' instead of 'did'.
    """
    if   did == "P4a": return "Z3"
    elif did == "P4p": return "Z2"
    elif did == "P0a": return "AB"
    else: return did


track_files = ["./data/embryo1/tracks.txt", "./data/embryo2/tracks.txt", "./data/embryo3/tracks.txt"]
time_cutoffs = [250, 242, 221]
valid_cell_names_union = set()
valid_cell_names_list = []

for track_file, time_cutoff in zip(track_files, time_cutoffs):
    ts_df = pd.read_csv(track_file, sep="\t")
    ts_df = ts_df.loc[ts_df["t"] <= time_cutoff]
    cell_names = ts_df["name"].unique()
    valid_cell_names = []
    for name in cell_names:
        time_points = ts_df.loc[ts_df['name'] == name]["t"].values
        if len(time_points) == 1 and time_points[0] == time_cutoff:
            continue
        valid_cell_names.append(name)
    valid_cell_names_union.update(valid_cell_names)
    valid_cell_names_list.append(valid_cell_names)
    
def load_json(file_path):
    """
    Load a JSON file and return its content as a Python object.
    
    :param file_path: Path to the JSON file.
    :return: Parsed JSON content as a Python object.
    """
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)
    
lineage_data = load_json('./data/cell_lineage.json')

terminal_nodes = []
terminal_parents = []
terminal_graph = defaultdict(list)
def dfs(node, parent):
    children = node.get("children", [])
    if len(children) == 0:
        lookup_name = map_names(node["did"])
        p_lookup_name = map_names(parent['did'])
        if lookup_name in valid_cell_names:
            terminal_nodes.append(lookup_name)
            terminal_parents.append(p_lookup_name)
            terminal_graph[p_lookup_name].append(lookup_name)
    else:
        for child in children:
            dfs(child, node)

dfs(lineage_data, None)

In [57]:
valid_cell_names_union_list = list(valid_cell_names_union)

In [ ]:
# go through each protein files in "data/protein" 
# and check if the lineage names are in the sample lineage names
# and average expression of each lineage across time points in protein data has full lineage coverage

# covered_proteins = 0
# lineage_raw_expression_mat = []
# lineage_adj_expression_mat = []
# protein_names = []
# for file in tqdm(os.listdir("data/protein")):
#     if file.endswith(".csv"):
#         protein_name = file.split('.')[0]
#         df = pd.read_csv(os.path.join("data/protein", file))
#         lineage_names = set(df['Cell-name'].unique())
#         lineage_raw_expression = []
#         lineage_adj_expression = []
#         covered_lineage_names = lineage_names.intersection(set(valid_cell_names_list[1]))
#         if len(covered_lineage_names) == len(valid_cell_names_list[1]):
#             covered_proteins += 1
#             protein_names.append(protein_name)
#             for name in valid_cell_names_list[0]:
#                 lineage_df = df[df['Cell-name'] == name]
#                 lineage_raw_expression.append(lineage_df['Raw-expression'].mean())
#                 lineage_adj_expression.append(lineage_df['Adjustment-expression'].mean())
#             lineage_raw_expression_mat.append(lineage_raw_expression)
#             lineage_adj_expression_mat.append(lineage_adj_expression)
        

100%|██████████| 292/292 [08:10<00:00,  1.68s/it]


In [81]:
# go through each protein files in "data/protein" 
# and check if the lineage names are in the sample lineage names
# and average expression of each lineage across time points in protein data has full lineage coverage

covered_proteins = 0
lineage_raw_expression_mat = []
lineage_adj_expression_mat = []
protein_names = []
for file in tqdm(os.listdir("data/protein")):
    if file.endswith(".csv"):
        protein_name = file.split('.')[0]
        df = pd.read_csv(os.path.join("data/protein", file))
        lineage_names = set(df['Cell-name'].unique())
        lineage_raw_expression = []
        lineage_adj_expression = []
        covered_lineage_names = lineage_names.intersection(set(valid_cell_names_union_list))
        covered_proteins += 1
        protein_names.append(protein_name)
        for name in valid_cell_names_union_list:
            lineage_df = df[df['Cell-name'] == name]
            if lineage_df.empty:
                lineage_raw_expression.append(np.nan)
                lineage_adj_expression.append(np.nan)
                continue
            lineage_raw_expression.append(lineage_df['Raw-expression'].mean())
            lineage_adj_expression.append(lineage_df['Adjustment-expression'].mean())
        lineage_raw_expression_mat.append(lineage_raw_expression)
        lineage_adj_expression_mat.append(lineage_adj_expression)
        

100%|██████████| 292/292 [10:17<00:00,  2.12s/it]


In [82]:
# fill nan with 0
lineage_raw_expression_mat = np.nan_to_num(np.array(lineage_raw_expression_mat), nan=0.0)
lineage_adj_expression_mat = np.nan_to_num(np.array(lineage_adj_expression_mat), nan=0.0)

In [83]:
# check number of zeros in each column
lineage_raw_expression_zeros = (lineage_raw_expression_mat == 0).sum(axis=0)
lineage_adj_expression_zeros = (lineage_adj_expression_mat == 0).sum(axis=0)

In [86]:
lineage_raw_expression_df = pd.DataFrame(lineage_raw_expression_mat, columns=valid_cell_names_union_list, index=protein_names)
lineage_adj_expression_df = pd.DataFrame(lineage_adj_expression_mat, columns=valid_cell_names_union_list, index=protein_names)
lineage_raw_expression_df.to_csv('data/protein/aggregated/lineage_raw_expression.csv')
lineage_adj_expression_df.to_csv('data/protein/aggregated/lineage_adj_expression.csv')